In [2]:
import pandas as pd
import re

In [2]:
##Load the train dataset
df_train = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

C:\Users\Admin\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [76]:
df_train.head(15)

,passage
id,
9797,"New data on viruses isolated from patients with subacute thyroiditis de Quervain \nare reported. Characteristic morphological, cytological, some physico-chemical \nand biological features of the isolated viruses are described. A possible role \nof these viruses in human and animal health disorders is discussed. The isolated \nviruses remain unclassified so far."
11906,"We describe an improved method for detecting deficiency of the acid hydrolase, \nalpha-1,4-glucosidase in leukocytes, the enzyme defect in glycogen storage \ndisease Type II (Pompe disease). The procedure requires smaller volumes of blood \nand less time than previous methods. The assay involves the separation of \nleukocytes by Peter's method for beta-glucosidase and a modification of Salafsky \nand Nadler's fluorometric method for alpha-glucosidase."
16083,"We have studied the effects of curare on responses resulting from iontophoretic \napplication of several putative neurotransmitters onto Aplysia neurons. These \nneurons have specific receptors for acetylcholine (ACh), dopamine, octopamine, \nphenylethanolamine, histamine, gamma-aminobutyric acid (GABA), aspartic acid, \nand glutamic acid. Each of these substances may on different specific neurons \nelicit at least three types of response, caused by a fast depolarizing Na+, a \nfast hyperpolarizing Cl-, or a slow hyperpolarizing K+ conductance increase. All \nresponses resulting from either Na+ or Cl- conductance increases, irrespective \nof which putative transmitter activated the response, were sensitive to curare. \nMost were totally blocked by less than or equal to 10-4 M curare. GABA responses \nwere less sensitive and were often only depressed by 10-3 M curare. K+ \nconductance responses, irrespective of the transmitter, were not curare \nsensitive. These results are consistent with a model of receptor organization in \nwhich one neurotransmitter receptor may be associated with any of at least three \nionophores, mediating conductance increase responses to Na+, Cl-, and K+, \nrespectively. In Aplysia nervous tissue, curare appears not to be a specific \nantagonist for the nicotinic ACh receptor, but rather to be a specific blocking \nagent for a class of receptor-activated Na+ and Cl- responses."
23188,"Kinetic and electrophoretic properties of 230--300 fold purified preparations of \nglucose-6-phosphate dehydrogenase (G6PD) from red cells of donors and patients \nwith acute drug hemolytic anemia due to G6PD deficiency were studied. A new \nabnormal variant of G6PD isolated from red cell of a patient with acute drug \nhemolytic anemia, which was not described in literature, has been discovered. \nThe abnormal enzyme differs from the normal by decreased Michaelis constant for \nglucose-6-phosphate and nicotinamide adenine dinucleotide phosphate (NADP), by \nincreased utilization of analogues of substrates--2-deoxy-glucose-6-phosphate \nand particularly deamino-NADP, by low thermal stability, by the character of \npH-dependence, by the appearance of a single band of G6PD activity in \npolyacrylamide gel electrophoresis."
23469,"Male Wistar specific-pathogen-free rats aged 2, 7, 17, 30, 60, 120, 200, 360 and \n600 days, all killed in experiment on the same day, were examined. The body \nweight significantly increased until the 200th day, the weight of adrenals until \nthe 120th day and the adrenal protein content until the 30th day of life. The \nadrenaline content of the adrenals increased continuously during the 600 days \nstudied. Adrenal noradrenaline content increased rapidly over the first 17 days, \nremained at a stable level until the 120th day, and rose to a higher level after \n200 days. The activity of adrenal catecholamine-synthesizing enzymes also \nincreased with age: tyrosine hydroxylase gradually increased until the 360th \nday, dopamine-beta-hydroxylase and phenylethanolamine-N-methyl transferase until \nthe 200th day. Our results demonstrate that, in the rat, during d

In [3]:
train = df_train.copy()

In [4]:
# Step 1: Define invalid values
def is_invalid_passage(val):
    return pd.isna(val) or str(val).strip().lower() in ['', 'nan']

# Step 2: Apply to your dataframe
invalid_mask = train['passage'].apply(is_invalid_passage)
num_invalid_rows = invalid_mask.sum()
print(f"Number of invalid (empty/'nan'/NaN) rows: {num_invalid_rows}")

# Step 3: Remove them
train_cleaned = train[~invalid_mask].reset_index(drop=True)
print(f"Shape after cleaning: {train_cleaned.shape}")



Number of invalid (empty/'nan'/NaN) rows: 12220
Shape after cleaning: (28001, 1)


In [80]:
# Show first 10 invalid rows
train[invalid_mask].head(10)

,passage
id,
97949,nan
98518,nan
100785,nan
117628,nan
125891,nan
227209,nan
227871,nan
261962,nan
355046,nan


In [81]:
train[invalid_mask].to_csv("invalid_passage_rows.csv", index=False)

In [82]:
train_cleaned.to_csv("train_cleaned_dataset_biosq.csv")

In [5]:
train_cleaned.head(3)

,passage
0,New data on viruses isolated from patients wit...
1,We describe an improved method for detecting d...
2,We have studied the effects of curare on respo...


In [ ]:
##Checking original text column contains patterns like emoji, URL, mentions, hashtags, etc.

In [6]:
import re
import pandas as pd

def check_preprocessing_targets(df, text_col='Tweet', slang_dict=None):
    """
    Checks if original text column contains patterns like emoji, URL, mentions, hashtags, etc.
    Returns a summary count for each pattern.
    """

    url_pattern = re.compile(r"http\S+|www\S+|https\S+")
    mention_pattern = re.compile(r"@\w+")
    hashtag_pattern = re.compile(r"#\w+")
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F"
             u"\U0001F300-\U0001F5FF"
             u"\U0001F680-\U0001F6FF"
             u"\U0001F1E0-\U0001F1FF" "]+", flags=re.UNICODE)
    number_pattern = re.compile(r"\b\d+\b") # Remove only standalone numbers
    date_pattern = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
    duration_pattern = re.compile(r"\b\d+\s?(hours?|days?|weeks?|mins?|minutes?|24/7)\b", flags=re.IGNORECASE)
    rating_pattern = re.compile(r"\b\d{1,2}/10\b")
    underscore_pattern = re.compile(r"_")

    def contains_slang(text):
        if slang_dict is None:
            return False
        return any(word in slang_dict for word in str(text).split())

    summary = {
        'URLs': df[text_col].apply(lambda x: bool(url_pattern.search(str(x)))).sum(),
        'Mentions (@)': df[text_col].apply(lambda x: bool(mention_pattern.search(str(x)))).sum(),
        'Hashtags (#)': df[text_col].apply(lambda x: bool(hashtag_pattern.search(str(x)))).sum(),
        'Emojis': df[text_col].apply(lambda x: bool(emoji_pattern.search(str(x)))).sum(),
        'Numbers': df[text_col].apply(lambda x: bool(number_pattern.search(str(x)))).sum(),
        'Dates': df[text_col].apply(lambda x: bool(date_pattern.search(str(x)))).sum(),
        'Durations': df[text_col].apply(lambda x: bool(duration_pattern.search(str(x)))).sum(),
        'Ratings (/10)': df[text_col].apply(lambda x: bool(rating_pattern.search(str(x)))).sum(),
        'Underscores': df[text_col].apply(lambda x: bool(underscore_pattern.search(str(x)))).sum(),
        'Slang': df[text_col].apply(contains_slang).sum() if slang_dict else 'Skipped',
    }

    return pd.DataFrame.from_dict(summary, orient='index', columns=['Count']).sort_values(by='Count', ascending=False)

     

In [7]:
slang_dict = {
    "DNA": "Deoxyribonucleic acid",
    "RNA": "Ribonucleic acid",
    "PCR": "Polymerase chain reaction",
    "IL": "Interleukin",
    "HIV": "Human immunodeficiency virus",
    "AD": "Alzheimer's disease",
    "EGFR": "Epidermal growth factor receptor",
    "TNF": "Tumor necrosis factor",
    "ALS": "Amyotrophic lateral sclerosis",
    "SLE": "Systemic lupus erythematosus",
    "MRI": "Magnetic resonance imaging",
    "CSF": "Cerebrospinal fluid",
    "MS": "Multiple sclerosis",
    "ER": "Estrogen receptor" 
}
     


In [8]:
check_preprocessing_targets(train_cleaned, text_col='passage', slang_dict=slang_dict)

,Count
Numbers,21942
Slang,5780
Durations,2675
URLs,486
Mentions (@),438
Underscores,170
Hashtags (#),63
Ratings (/10),40
Dates,31
Emojis,0


In [17]:
def clean_tweet(tweet):
    tweet = str(tweet).lower()#Converts the tweet to lowercase
    tweet = re.sub(r"@\S+", '', tweet)  # remove mentions
    tweet = re.sub(r'\b10\^-(\d+)\b', r'1e-\1', tweet) # convert 10^-3 to 1e-3
    tweet = re.sub(r"http\S+|www\S+|https\S+", '', tweet)  # remove URLs
    tweet = re.sub(r"#", '', tweet)  # remove hashtag #
    tweet = re.sub(r"[^\w\s']", ' ', tweet)  # keep words, spaces, and apostrophes
    tweet = re.sub(r'\s+', ' ', tweet).strip()  # remove extra spaces
    return tweet

In [12]:
#cheeck
index = 2  # Pick any index you want
original_passage =train_cleaned.loc[index, 'passage']
cleaned_passage = clean_tweet(original_passage)

print(f"Original Tweet: {original_passage}")
print(f"Cleaned Tweet : {cleaned_passage}")


Original Tweet: We have studied the effects of curare on responses resulting from iontophoretic 
application of several putative neurotransmitters onto Aplysia neurons. These 
neurons have specific receptors for acetylcholine (ACh), dopamine, octopamine, 
phenylethanolamine, histamine, gamma-aminobutyric acid (GABA), aspartic acid, 
and glutamic acid. Each of these substances may on different specific neurons 
elicit at least three types of response, caused by a fast depolarizing Na+, a 
fast hyperpolarizing Cl-, or a slow hyperpolarizing K+ conductance increase. All 
responses resulting from either Na+ or Cl- conductance increases, irrespective 
of which putative transmitter activated the response, were sensitive to curare. 
Most were totally blocked by less than or equal to 10-4 M curare. GABA responses 
were less sensitive and were often only depressed by 10-3 M curare. K+ 
conductance responses, irrespective of the transmitter, were not curare 
sensitive. These results are consiste

**The function performs well for general text cleaning but distorts important scientific symbols and notations. For use with a causal transformer on biomedical or technical content, modify the regex to preserve characters like +, -, . and ^**
**Keep periods, as they preserve decimal values, scientific notation, and sentence structure essential for accurate processing in scientific texts.**

In [20]:
import re

def clean_tweet(tweet):
    tweet = str(tweet).lower()  # Convert to lowercase
    tweet = re.sub(r"@\S+", '', tweet)  # Remove mentions
    tweet = re.sub(r"http\S+|www\S+|https\S+", '', tweet)  # Remove URLs
    tweet = re.sub(r"#", '', tweet)  # Remove hashtag symbol #

    # Preserve important symbols: +, -, ., ^, = and apostrophes
    tweet = re.sub(r"[^\w\s'\+\-\.\^=]", ' ', tweet)

    # Convert 10-3 and 10^-3 to Python-readable format (1e-3)
    tweet = re.sub(r'\b10\^-(\d+)\b', r'1e-\1', tweet)  # 10^-3 → 1e-3
    tweet = re.sub(r'\b10-(\d+)\b', r'1e-\1', tweet)    # 10-3 → 1e-3

    tweet = re.sub(r'\s+', ' ', tweet).strip()  # Remove extra spaces
    return tweet


In [23]:
#cheeck
index = 2  # Pick any index you want
original_passage =train_cleaned.loc[index, 'passage']
cleaned_passage = clean_tweet(original_passage)

print(f"Original Tweet: {original_passage}")
print(f"Cleaned Tweet : {cleaned_passage}")

Original Tweet: We have studied the effects of curare on responses resulting from iontophoretic 
application of several putative neurotransmitters onto Aplysia neurons. These 
neurons have specific receptors for acetylcholine (ACh), dopamine, octopamine, 
phenylethanolamine, histamine, gamma-aminobutyric acid (GABA), aspartic acid, 
and glutamic acid. Each of these substances may on different specific neurons 
elicit at least three types of response, caused by a fast depolarizing Na+, a 
fast hyperpolarizing Cl-, or a slow hyperpolarizing K+ conductance increase. All 
responses resulting from either Na+ or Cl- conductance increases, irrespective 
of which putative transmitter activated the response, were sensitive to curare. 
Most were totally blocked by less than or equal to 10-4 M curare. GABA responses 
were less sensitive and were often only depressed by 10-3 M curare. K+ 
conductance responses, irrespective of the transmitter, were not curare 
sensitive. These results are consiste

In [78]:
import re

def biomedical_clean_tweet_final(text):
    text = str(text).lower()

    # ================================
    # 1. Protect biomedical numbers + units
    # ================================

    protected_patterns = [
        r'\b\d+\.\d+\b',                                           # decimal: 24.5
        r'\b\d+(\.\d+)?\s?(?:h|hr|hrs|hours?|mins?|minutes?)\b',   # durations like 15 min, 24.5 h
        r'\b\d+(\.\d+)?\s?(?:days?|weeks?|months?|times?)\b',      # 3 days, 7 weeks
        r'\b\d+\s?(?:-day)\b',                                     # 28-day
        r'\b\d+\s?(?:degrees?\s?c|°c)\b',                          # temperature
        r'\b\d+(\.\d+)?\s?(?:mg\/kg|mg|g|kg|ml|l|mm|mM|µg|ug|μg)\b',# biomedical units
        r'\b\w+-\w+\b',                                            # hyphenated biomedical terms (e.g., wr-1065)
        r'\b\d+\)',                                                # groupings like 1)
    ]

    # Protect comma-separated lists like "1, 3, and 7 days"
    list_pattern = r'((?:\d+\s*,\s*)+\d+)(?=\s*(?:days?|weeks?|hours?|minutes?))'
    text = re.sub(list_pattern, lambda m: f"__LIST__{m.group(1)}__", text)

    for i, pattern in enumerate(protected_patterns):
        text = re.sub(pattern, lambda m: f"__PROT{i}__{m.group(0)}__", text, flags=re.IGNORECASE)

    # ================================
    # 2. Remove standalone numbers
    # ================================
    text = re.sub(r'\b\d+\b', '', text)

    # ================================
    # 3. Restore protected values
    # ================================
    text = re.sub(r'__PROT\d+__(.*?)__', r'\1', text)
    text = re.sub(r'__LIST__(.*?)__', r'\1', text)

    # ================================
    # 4. Final cleanup
    # ================================
    text = re.sub(r"[^\w\s\.\-\+\^\=\/\(\)]", ' ', text)  # preserve scientific symbols
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [46]:
from symspellpy import SymSpell, Verbosity
import importlib.resources

# Initialize SymSpell only once (outside the function)
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = importlib.resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
sym_spell.load_dictionary(str(dictionary_path), term_index=0, count_index=1)

def correct_text_spelling(text):
    try:
        words = text.split()
        corrected_words = []
        for word in words:
            if "'" in word:  # skip contractions like "don't", "I'm"
                corrected_words.append(word)
            else:
                suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
                corrected = suggestions[0].term if suggestions else word
                corrected_words.append(corrected)
        return ' '.join(corrected_words)
    except Exception:
        return text

In [47]:
#cheeck
index = 2  # Pick any index you want
original_passage =train_cleaned.loc[index, 'passage']
cleaned_passage = correct_text_spelling(original_passage)

print(f"Original Tweet: {original_passage}")
print(f"Cleaned Tweet : {cleaned_passage}")

Original Tweet: We have studied the effects of curare on responses resulting from iontophoretic 
application of several putative neurotransmitters onto Aplysia neurons. These 
neurons have specific receptors for acetylcholine (ACh), dopamine, octopamine, 
phenylethanolamine, histamine, gamma-aminobutyric acid (GABA), aspartic acid, 
and glutamic acid. Each of these substances may on different specific neurons 
elicit at least three types of response, caused by a fast depolarizing Na+, a 
fast hyperpolarizing Cl-, or a slow hyperpolarizing K+ conductance increase. All 
responses resulting from either Na+ or Cl- conductance increases, irrespective 
of which putative transmitter activated the response, were sensitive to curare. 
Most were totally blocked by less than or equal to 10-4 M curare. GABA responses 
were less sensitive and were often only depressed by 10-3 M curare. K+ 
conductance responses, irrespective of the transmitter, were not curare 
sensitive. These results are consiste

**For domain-specific corpora like this one, skipping spelling correction entirely may be better than applying SymSpell blindly.

You can conditionally apply correction only if the document is non-technical.**



In [64]:
import spacy
nlp = spacy.load("en_core_sci_sm")

def handle_negation_enhanced(text):
    """
    Adds 'NOT_' prefix to negated heads and surrounding tokens
    like intensifiers or modifiers within a limited scope.
    """
    doc = nlp(text)
    negated_indices = set()

    for token in doc:
        if token.dep_ == "neg":
            head = token.head
            negated_indices.add(head.i)

            # Include immediate modifiers or right-hand phrase
            for right in head.rights:
                if right.dep_ in {"advmod", "amod", "dobj", "attr", "prep"} or right.pos_ in {"ADJ", "NOUN", "VERB"}:
                    negated_indices.add(right.i)

            # Optionally include intensifiers before the verb/adjective
            for child in head.children:
                if child.dep_ in {"advmod", "amod"}:
                    negated_indices.add(child.i)

    # Build modified sentence
    result = []
    for i, token in enumerate(doc):
        if i in negated_indices:
            result.append("NOT_" + token.text)
        else:
            result.append(token.text)

    return " ".join(result)


In [66]:
#POS/NER annotation function
import spacy
import pandas as pd
from tqdm import tqdm

# Load biomedical NER + POS model
nlp = spacy.load("en_core_sci_sm")  # Includes NER + POS tagging

# Disable components if you only want NER/POS
# nlp.disable_pipes(['lemmatizer'])  # Optional

def annotate_text(text):
    if not isinstance(text, str) or not text.strip():
        return [], []

    doc = nlp(text)
    ents = [(ent.text, ent.label_) for ent in doc.ents]
    pos = [(token.text, token.pos_) for token in doc]
    return ents, pos


In [67]:
train_sampled = train_cleaned.sample(n=10000, random_state=42).reset_index(drop=True)

In [68]:
#Apply the above functions
train_sampled['Cleaned_passage'] = train_sampled['passage'].apply(clean_tweet)

In [79]:
# Apply the biomedical_clean_tweet_final function to the cleaned passages
train_sampled['Cleaned_passage_2'] = train_sampled['Cleaned_passage'].apply(biomedical_clean_tweet_final)


In [74]:
# Set display options to show full text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

In [80]:
# Display first sample
row = train_sampled.iloc[0]

print(f"Original Tweet       : {row['passage']}")
print(f"Cleaned Tweet (Step1): {row['Cleaned_passage']}")
print(f"Cleaned Tweet (Step2): {row['Cleaned_passage_2']}")


Original Tweet       : The objectives of this study were to evaluate the protective effects of 
amifostine against paclitaxel-induced toxicity to normal and malignant human 
tissues. Haematopoietic progenitor colony assays were used to establish the 
number of CFU-GEMM and BFU-E colonies after incubation with WR-1065 alone, 
Amifostine alone, paclitaxel (2.5 or 5 microM) +/- WR-1065 or amifostine. MTT 
and alkaline elution assays evaluated the in vitro growth inhibitory and DNA 
damaging effects, respectively, of paclitaxel with or without amifostine against 
normal human fibroblasts and human non-small cell lung cancer (NSCLC) cells. 
This combination was also evaluated in vivo using severe combined immune 
deficient (scid) mouse models of early (non-palpable tumours) and advanced 
(palpable tumours) human ovarian cancer. Human 2780 ovarian cancer cells were 
inoculated subcutaneously while paclitaxel and amifostine were administered 
intraperitoneally. A brief exposure (15 min) to am

In [81]:
# Apply the negation handling function to Cleaned_passage_2
train_sampled['final_Cleaned_passage'] = train_sampled['Cleaned_passage_2'].apply(handle_negation_enhanced)

In [83]:
# Apply the annotation function to the final cleaned passage
tqdm.pandas()  # Enable progress bar for apply with tqdm

# Create new columns for NER and POS
train_sampled[['NER_tags', 'POS_tags']] = train_sampled['final_Cleaned_passage'].progress_apply(
    lambda x: pd.Series(annotate_text(x))
)


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [05:52<00:00, 28.37it/s]


In [85]:
from spacy.lang.en.stop_words import STOP_WORDS
from tqdm import tqdm

# Biomedical terms to always keep (you can expand this list)
protected_terms = {
    "nsclc", "cfu-gemm", "bfu-e", "wr-1065", "wr", "mtx", "mtt", "dna", "rna",
    "iv", "in vitro", "in vivo", "scid", "mg", "kg", "μm", "mm", "cm", "ml"
}

# Normalize protected terms (lowercase for matching)
protected_terms = set(term.lower() for term in protected_terms)

tqdm.pandas()

def biomedical_lemmatize(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        lemma = token.lemma_.lower()
        original = token.text.lower()
        if (
            original in protected_terms or
            token.text in protected_terms or
            token.ent_type_ != "" or   # keep biomedical entities
            token.text.isupper()       # preserve acronyms
        ):
            tokens.append(token.text)  # keep original form
        elif lemma not in STOP_WORDS and token.is_alpha and len(lemma) > 1:
            tokens.append(lemma)
    return " ".join(tokens)

# Apply to column
train_sampled['lemmatized_passage'] = train_sampled['final_Cleaned_passage'].progress_apply(biomedical_lemmatize)


100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [05:04<00:00, 32.89it/s]


In [87]:
train_sampled.to_csv("train_preprocessed_biomedical.csv", index=False)


In [88]:
test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")


In [89]:
test.head(1)

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multifactorial disorder?,"Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.","[20598273, 6650562, 15829955, 15617541, 23001136, 8896569, 21995290, 12239580, 15858239]"


In [90]:
from tqdm import tqdm
tqdm.pandas()

# Final full pipeline function for test set
def preprocess_biomedical_text(text):
    if not isinstance(text, str):
        return ""
    t1 = clean_tweet(text)
    t2 = biomedical_clean_tweet_final(t1)
    t3 = handle_negation_enhanced(t2)
    t4 = biomedical_lemmatize(t3)
    return t4

# Apply to both columns
test['question_clean'] = test['question'].progress_apply(preprocess_biomedical_text)
test['answer_clean'] = test['answer'].progress_apply(preprocess_biomedical_text)


100%|██████████████████████████████████████████████████████████████████████████████| 4719/4719 [01:20<00:00, 58.59it/s]


In [91]:
test.head(1)

,question,answer,relevant_passage_ids,question_clean,answer_clean
id,,,,,
0,Is Hirschsprung disease a mendelian or a multifactorial disorder?,"Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model.","[20598273, 6650562, 15829955, 15617541, 23001136, 8896569, 21995290, 12239580, 15858239]",hirschsprung disease mendelian multifactorial disorder,coding sequence mutations ret gdnf ednrb edn3 sox10 involve development hirschsprung disease majority genes related mendelian syndromic form hirschsprung disease non-mendelian inheritance sporadic non-syndromic hirschsprung disease prove complex involvement multiple loci demonstrate multiplicative model


In [92]:
test.to_csv("test_processed.csv", index=False)
